# 02 Material balance: the Rough tank for methane and hydrogen
Purpose: check `reservoir.py` (closed-tank p/Z material balance, decisions A2, A8–A10), test it against Rough's historic methane operation, and give a **static** upper bound on hydrogen working gas to compare with published estimates (validation step V4 in `nodal_model_design.md`).

Static means the wells and surface facilities are assumed able to move the reservoir across the whole pressure window. The later notebooks remove that assumption.

In [ ]:
# Paste as the first cell of every notebook
import subprocess, hashlib, datetime, pathlib
root = pathlib.Path.cwd().parent
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=root).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=root).stdout.strip() != ""
inp = hashlib.sha256((root / "data" / "inputs.csv").read_bytes()).hexdigest()[:10]
print(f"Run {datetime.datetime.now():%Y-%m-%d %H:%M} | commit {commit}{' (UNCOMMITTED CHANGES)' if dirty else ''} | inputs.csv {inp}")

## 1. Build the tank
The gas pore volume is fixed from the methane GIIP at the initial pressure (Scafidi et al., 2023, Table 1) and reused for hydrogen (A9). The first check is a round trip: a methane tank at p_i must hold the GIIP.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from h2_bottleneck import inputs, props
from h2_bottleneck.reservoir import Tank, working_gas

x = inputs.load()
T, p_i = x["reservoir_temperature"], x["reservoir_pressure_initial"]
HHV = {"Hydrogen": x["hhv_hydrogen"], "Methane": x["hhv_methane"]}   # J/kg, Scafidi et al. (2023) Table 2
TWH = 3.6e15            # J per TWh
BCF = 28316.85e3        # Sm3 per bcf (60 degF basis, ~0.2 % offset, decision X2)
DAY = 86_400.0

full = Tank.from_inputs(x, "Methane")
V = full.V
print(f"Gas pore volume V = {V/1e6:.1f} million m3")
print(f"Check: CH4 tank at p_i holds {full.inventory_sc/BCF:.1f} bcf (GIIP 366 bcf) "
      f"at {full.p/1e5:.1f} bar (p_i = {p_i/1e5:.1f} bar)")

## 2. p/Z plot and energy in place
For a closed tank, p/Z is proportional to the gas in place. On a molar (standard-volume) basis the line is the same for both gases, so any difference between them comes only from Z at a given pressure. The energy penalty appears when inventory is converted to energy.

In [ ]:
P = np.linspace(20e5, 320e5, 61)
rows = []
for gas in HHV:
    for p in P:
        t = Tank.at_pressure(V, p, T, gas)
        rows.append(dict(gas=gas, p_bar=p/1e5, pZ_bar=p/t.Z/1e5,
                         inv_bcf=t.inventory_sc/BCF, E_TWh=t.energy(HHV[gas])/TWH))
mb = pd.DataFrame(rows)

# Linearity check: (p/Z)/inventory should be constant for each gas
for gas, g in mb.groupby("gas"):
    k = g.pZ_bar / g.inv_bcf
    print(f"{gas:8s}: (p/Z)/inventory varies by {(k.max()/k.min()-1)*100:.1e} % over 20-320 bar")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for gas, g in mb.groupby("gas"):
    ax[0].plot(g.inv_bcf, g.pZ_bar, label=gas)
    ax[1].plot(g.inv_bcf, g.p_bar, label=gas)
    ax[2].plot(g.E_TWh, g.p_bar, label=gas)
labels = [("Gas in place (bcf)", "p/Z (bar)"),
          ("Gas in place (bcf)", "Reservoir pressure (bar)"),
          ("Energy in place, HHV (TWh)", "Reservoir pressure (bar)")]
for a, (xl, yl) in zip(ax, labels):
    a.set_xlabel(xl); a.set_ylabel(yl); a.legend()
ax[0].set_title("(a) p/Z", loc="left"); ax[1].set_title("(b) Pressure", loc="left")
ax[2].set_title("(c) Energy", loc="left")
fig.tight_layout(); fig.savefig("../results/figures/02_material_balance.png", dpi=300)

## 3. Calibration against Rough's methane operation
Centrica (2006, p.6) gives about 118 bcf working gas and 185 bcf cushion gas. Putting those inventories into the tank gives the reservoir pressure window that historic operation implies. It should sit below p_i and roughly match the BHP limits in Heinemann et al. (2025, Table 1): maximum 282–306 bar, minimum 98–185 bar per well.

In [ ]:
def p_at_inventory(gas, inv_sc):
    """Average reservoir pressure (Pa) when the tank holds inv_sc (Sm3) of gas."""
    return Tank(V, T, gas, inv_sc * props.rho_sc(gas)).p

p_ch4_low = p_at_inventory("Methane", 185 * BCF)
p_ch4_high = p_at_inventory("Methane", (185 + 118) * BCF)
print(f"Implied CH4 window: {p_ch4_low/1e5:.0f} bar (cushion only) to {p_ch4_high/1e5:.0f} bar (full)")
print(f"Total 303 bcf is {303/366*100:.0f} % of GIIP")

## 4. Static working gas
Working and cushion gas for three pressure windows, for both gases:
- **Heinemann cycle:** average reservoir pressure about 180–275 bar in their hydrogen runs (Heinemann et al., 2025, Fig. 4).
- **Historic CH4 window:** from Section 3.
- **Full:** 180 bar to p_i.

Published hydrogen figures for Rough: 7.5 TWh working and 16.0 TWh cushion (Heinemann et al., 2025); 10–15 TWh (Cavanagh et al., 2022); 10–16 TWh (Centrica, cited in Heinemann et al., 2025); about 42 % of the methane energy capacity (Amid et al., 2016).

In [ ]:
windows = {
    "Heinemann H2 cycle": (180e5, 275e5),
    "Historic CH4 window": (p_ch4_low, p_ch4_high),
    "180 bar to p_i": (180e5, p_i),
}
rows = []
for name, (lo, hi) in windows.items():
    row = {"Window": name, "p_low (bar)": round(lo/1e5), "p_high (bar)": round(hi/1e5)}
    for gas, tag in (("Hydrogen", "H2"), ("Methane", "CH4")):
        w, c = working_gas(V, T, gas, lo, hi)
        row[f"{tag} working (TWh)"] = round(w * HHV[gas] / TWH, 2)
        row[f"{tag} cushion (TWh)"] = round(c * HHV[gas] / TWH, 2)
        row[f"{tag} cushion/working"] = round(c / w, 2)
        row[f"{tag} working (bcf)"] = round(w / props.rho_sc(gas) / BCF, 1)
    row["H2/CH4 working energy"] = round(row["H2 working (TWh)"] / row["CH4 working (TWh)"], 3)
    rows.append(row)
wg = pd.DataFrame(rows).set_index("Window")
wg.to_csv("../results/tables/02_static_working_gas.csv")
wg.T

## 5. Average rates the cycle would need
If the Heinemann working gas is moved on their schedule (4 months injection, 3 months withdrawal), the average standard rates below are needed. Compare with Rough's methane rates: about 15 million Sm3/d average injection and 44.8 million Sm3/d maximum withdrawal (Centrica, 2006). These are averages from the tank alone, not deliverability; the well and surface models decide whether they are achievable.

In [ ]:
lo, hi = windows["Heinemann H2 cycle"]
w_h2, _ = working_gas(V, T, "Hydrogen", lo, hi)
w_sc = w_h2 / props.rho_sc("Hydrogen")
q_inj = w_sc / (120 * DAY)          # Sm3/s
q_wd = w_sc / (90 * DAY)
print(f"H2 working gas: {w_sc/1e9:.2f} billion Sm3 = {w_h2*HHV['Hydrogen']/TWH:.2f} TWh")
print(f"Average injection over 120 d:  {q_inj*DAY/1e6:.1f} million Sm3/d  (CH4 historic ~15)")
print(f"Average withdrawal over 90 d: {q_wd*DAY/1e6:.1f} million Sm3/d  (CH4 historic max 44.8)")

## 6. Time-stepping check (illustrative cycle)
Step the tank day by day at the constant average rates above: inject 120 days, rest 90, withdraw 90, rest 60 (Heinemann's 4/3/3/2-month schedule). The pressure should rise from 180 to 275 bar and return exactly to 180 bar. This checks `Tank.step`; it is not a deliverability result.

In [ ]:
schedule = [(+q_inj, 120), (0.0, 90), (-q_wd, 90), (0.0, 60)]
t = Tank.at_pressure(V, lo, T, "Hydrogen")
days, p_bar = [0], [t.p / 1e5]
for q, n in schedule:
    for _ in range(n):
        t = t.step(q, DAY)
        days.append(days[-1] + 1); p_bar.append(t.p / 1e5)
print(f"Start {p_bar[0]:.2f} bar, peak {max(p_bar):.2f} bar, end {p_bar[-1]:.2f} bar")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(days, p_bar)
ax.set_xlabel("Day of cycle"); ax.set_ylabel("Reservoir pressure (bar)")
ax.set_title("Illustrative H2 cycle at constant average rates", loc="left")
fig.tight_layout(); fig.savefig("../results/figures/02_illustrative_cycle.png", dpi=300)

## Findings

**Checks** 

The tank returns Rough's GIIP (366.0 bcf) at the initial pressure (312.5 bar), with a gas pore volume of 42.8 million m³. p/Z is linear in inventory for both gases to within 1.2 × 10⁻⁷ %. The illustrative cycle returns exactly to its starting pressure of 180 bar, so the mass balance in `Tank.step` is conserved. On a standard-volume basis the two gases share one p/Z line (Figure 02a), so the difference between them comes only from Z at a given pressure: hydrogen holds less gas than methane at the same pressure (02b) and much less energy (02c).

**Historic methane window** 

Centrica's 118 bcf working and 185 bcf cushion gas (Centrica, 2006) imply an average reservoir pressure of 147–248 bar. This lies inside Heinemann et al.'s (2025) per-well bottomhole limits (maximum 282–306 bar, minimum 98–185 bar), as expected, because injection needs bottomhole pressures above the reservoir average and withdrawal below it. The pore volume is therefore consistent with historic operation. Heinemann's hydrogen cycle (about 180–275 bar) operates the reservoir about 30 bar higher than historic methane operation did.

**Static hydrogen working gas** 

In the 180–275 bar window the tank gives 8.49 TWh of working gas and 18.44 TWh of cushion (ratio 2.17), against 7.5 and 16.0 TWh (ratio 2.1) from Heinemann et al. (2025). The working gas and cushion are both about 13–15 % higher, while the ratio agrees to within 3 %. A uniform offset of this kind points to a larger connected volume in the single tank, consistent with the compartmentalisation Heinemann describes (decision A9), rather than to well deliverability, which would reduce the working gas alone.

**Energy penalty** 

In the same pressure window, hydrogen's working gas holds 26.5–27.9 % of methane's energy; in the historic window that is 9.3 TWh against 35.0 TWh. Amid et al. (2016) report 42 % (12.6 against 30.2 TWh, lower heating value) because they do not hold the window fixed: working gas is set by what each gas can deliver over a 120-day withdrawal at a 5–10 MPa delivery pressure, giving about 50 % of the total gas as working gas for hydrogen against 30 % for methane. Their total stored energies agree with this tank to within about 3 % on the same heating-value basis, so the difference lies in the operating range, not the reservoir. The 42 % is therefore not a property of the gas: it assumes hydrogen can be drawn down much further than methane, which depends on the wells and on the delivery pressure at the surface, the constraints this project models. Published hydrogen figures of 10–16 TWh (Cavanagh et al., 2022; Centrica, cited in Heinemann et al., 2025) likewise exceed the static value for either fixed window. They are reached only if the reservoir is cycled over a wider range, for example from 180 bar to near its initial pressure (11.6 TWh), which is above any pressure historic operation reached.

**Rates** 

Moving 8.49 TWh on Heinemann's schedule needs an average injection of 21.1 million Sm³/d over 120 days. That is about 1.4 times Rough's historic average methane injection of about 15 million Sm³/d, although it carries only about 45 % of the energy (71 against 157 GWh/d). Compressor work scales with the molar (standard-volume) flow and the pressure ratio, so the existing compressors would have to handle about 40 % more gas, against a reservoir about 30 bar higher, with a lighter gas column in the wells. This is the first quantitative support for the hypothesis that compression limits injection (Section 3.2). Withdrawal needs an average of 28.1 million Sm³/d, 63 % of the historic peak of 44.8, which suggests withdrawal is less constrained. These are averages from the tank alone; the well, pipeline and compressor models decide whether they can be delivered.

**Limitations**
 
 These results are static: they assume the wells and surface facilities can move the reservoir across the whole window. The tank is a single compartment with no underburden support (A9), and GIIP carries a 0.2 % standard-conditions offset (X2).